# Personalized News Feed Recommender — MIND-small Prototype
#
Dataset: arashnic/mind-news-dataset, MINDsmall_train split only (no dev split
in this attachment) — so we carve our own train/validation split from it.
Each section below (# %%) is meant to be its own notebook cell.

## 0. Setup — installs

In [27]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

!pip install -q bertopic sentence-transformers implicit mlxtend umap-learn hdbscan scikit-learn

In [28]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict

import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import adjusted_rand_score, roc_auc_score

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

import implicit
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

## 1. Load & preprocess
#
Kept columns only (per our agreed spec):
- news.tsv: News ID, Title, Category (Category held aside for validation only)
- behaviors.tsv: Impression ID, User ID, Time, History, Impressions

In [29]:
BASE_PATH = "/kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train"

NEWS_PATH = f"{BASE_PATH}/news.tsv"
BEHAVIORS_PATH = f"{BASE_PATH}/behaviors.tsv"

assert os.path.isfile(NEWS_PATH), f"Not a file: {NEWS_PATH}"
assert os.path.isfile(BEHAVIORS_PATH), f"Not a file: {BEHAVIORS_PATH}"
print("Paths OK:", NEWS_PATH, BEHAVIORS_PATH)

Paths OK: /kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train/news.tsv /kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train/behaviors.tsv


In [30]:
news_cols = ["news_id", "category", "subcategory", "title", "abstract",
             "url", "title_entities", "abstract_entities"]
news_df = pd.read_csv(NEWS_PATH, sep="\t", header=None, names=news_cols)
news_df = news_df[["news_id", "title", "category"]].drop_duplicates("news_id").reset_index(drop=True)

behav_cols = ["impression_id", "user_id", "time", "history", "impressions"]
behav_df = pd.read_csv(BEHAVIORS_PATH, sep="\t", header=None, names=behav_cols)
behav_df["time"] = pd.to_datetime(behav_df["time"], format="%m/%d/%Y %I:%M:%S %p")

print(news_df.shape, behav_df.shape)

(51282, 3) (156965, 5)


In [31]:
def parse_history(h):
    if pd.isna(h):
        return []
    return h.split()

def parse_impressions(imp):
    out = []
    for tok in imp.split():
        nid, label = tok.rsplit("-", 1)
        out.append((nid, int(label)))
    return out

behav_df["history_list"] = behav_df["history"].apply(parse_history)
behav_df["impression_list"] = behav_df["impressions"].apply(parse_impressions)

### Train / validation split
Only the train split was attached, so we carve our own held-out slice
(most recent 10% of impressions by time) to keep evaluation honest.

In [32]:
behav_df = behav_df.sort_values("time").reset_index(drop=True)
split_idx = int(len(behav_df) * 0.9)
train_behav_df = behav_df.iloc[:split_idx].reset_index(drop=True)
val_behav_df = behav_df.iloc[split_idx:].reset_index(drop=True)
print(f"Train: {len(train_behav_df)} impressions, Val: {len(val_behav_df)} impressions")

Train: 141268 impressions, Val: 15697 impressions


In [33]:
# Flatten TRAIN impressions into a long (impression_id, user_id, news_id, label, time) table
def flatten_impressions(df):
    rows = []
    for _, r in df.iterrows():
        for nid, label in r["impression_list"]:
            rows.append((r["impression_id"], r["user_id"], nid, label, r["time"]))
    return pd.DataFrame(rows, columns=["impression_id", "user_id", "news_id", "label", "time"])

interactions_df = flatten_impressions(train_behav_df)
print(interactions_df.shape)
interactions_df.head()

(5235691, 5)


,impression_id,user_id,news_id,label,time
0,20112,U65916,N54300,0,2019-11-09 00:00:19
1,20112,U65916,N46057,1,2019-11-09 00:00:19
2,20112,U65916,N57005,0,2019-11-09 00:00:19
3,20112,U65916,N52154,0,2019-11-09 00:00:19
4,20112,U65916,N57099,0,2019-11-09 00:00:19


## 2. Unsupervised category discovery — BERTopic
Embeds Title only (Abstract intentionally excluded — see train/serve consistency note).

In [34]:
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
from umap import UMAP

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
titles = news_df["title"].fillna("").tolist()
title_embeddings = embed_model.encode(titles, show_progress_bar=True, batch_size=256)

# Custom vectorizer so topic keyword labels are real words, not stopwords
vectorizer_model = CountVectorizer(stop_words="english", min_df=2)

# Tuned UMAP + HDBSCAN — lower min_samples reduces the noise (-1) bucket
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=15, min_samples=5, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)

topic_model = BERTopic(embedding_model=embed_model, umap_model=umap_model,
                       hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
                       min_topic_size=15, calculate_probabilities=False)
cluster_ids, _ = topic_model.fit_transform(titles, title_embeddings)

print(f"Discovered {len(set(cluster_ids)) - (1 if -1 in cluster_ids else 0)} clusters "
      f"(+ noise bucket -1 if present)")
print("noise bucket size before reduction:", (np.array(cluster_ids) == -1).sum(), "/", len(cluster_ids))

# Reassign remaining noise-bucket articles to their nearest real topic
cluster_ids = topic_model.reduce_outliers(titles, cluster_ids, strategy="c-tf-idf")
topic_model.update_topics(titles, topics=cluster_ids)
news_df["cluster_id"] = cluster_ids

print("noise bucket size after reduction:", (news_df["cluster_id"] == -1).sum(), "/", len(news_df))
topic_model.get_topic_info().head(15)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/201 [00:00<?, ?it/s]

Discovered 633 clusters (+ noise bucket -1 if present)
noise bucket size before reduction: 19655 / 51282


2026-08-03 10:29:21,496 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


noise bucket size after reduction: 4 / 51282


,Topic,Count,Name,Representation,Representative_Docs
0,-1,4,-1_biopics_testosterone_tardigrade_rko,"[biopics, testosterone, tardigrade, rko, nowhe...",[Steelton woman charged in 2018 death of two-m...
1,0,995,0_ford_cars_corvette_mustang,"[ford, cars, corvette, mustang, chevy, 2020, s...",[Ford Reveals Acceleration Times for 760-HP Mu...
2,1,742,1_snow_cold_arctic_weather,"[snow, cold, arctic, weather, temperatures, wi...","[Get ready, Louisville: Here's what to know ab..."
3,2,675,2_impeachment_hearings_inquiry_democrats,"[impeachment, hearings, inquiry, democrats, pu...",[Impeachment inquiry begins public hearings: H...
4,3,608,3_dog_dogs_adopt_pet,"[dog, dogs, adopt, pet, puppy, cat, adoption, ...",[Looking to adopt a pet? Here are 5 lovable ki...
5,4,405,4_bruins_nhl_penguins_blackhawks,"[bruins, nhl, penguins, blackhawks, wings, cap...","[Pastrnak scores 10th, Bruins beat Maple Leafs..."
6,5,476,5_restaurant_restaurants_chef_food,"[restaurant, restaurants, chef, food, bar, caf...",[Chicago Has One of America's Best All-You-Can...
7,6,401,6_veterans_veteran_parade_honor,"[veterans, veteran, parade, honor, ceremony, d...",[Veterans Day Ceremony At The Carolina Field O...
8,7,327,7_meghan_prince_markle_royal,"[meghan, prince, markle, royal, kate, middleto...","[How Meghan Markle, Prince Harry, and the rest..."
9,8,338,8_ufc_diaz_244_bellator,"[ufc, diaz, 244, bellator, masvidal, fight, jo...",[Nate Diaz paid back fan who lost a bet on his...


In [37]:
# how many articles ended up unclustered?
print(news_df["cluster_id"].value_counts().head(10))
print("noise bucket size:", (news_df["cluster_id"] == -1).sum(), "/", len(news_df))

cluster_id
0     995
1     742
2     675
3     608
5     476
4     405
6     401
8     338
14    333
7     327
Name: count, dtype: int64
noise bucket size: 4 / 51282


## 3. Validate clusters against real Category (sanity check only — not used in training)

In [38]:
valid_mask = news_df["cluster_id"] != -1
ari = adjusted_rand_score(news_df.loc[valid_mask, "category"], news_df.loc[valid_mask, "cluster_id"])
print(f"Adjusted Rand Index vs real Category labels: {ari:.4f}")
pd.crosstab(news_df["cluster_id"], news_df["category"]).iloc[:15, :10]

Adjusted Rand Index vs real Category labels: 0.0124


category,autos,entertainment,finance,foodanddrink,health,kids,lifestyle,middleeast,movies,music
cluster_id,,,,,,,,,,
-1,0,0,0,0,1,0,0,0,1,0
0,846,1,46,1,1,1,5,0,2,3
1,14,0,5,1,2,2,6,0,0,1
2,0,0,2,0,0,0,1,0,0,1
3,1,4,0,4,13,1,185,0,3,2
4,0,0,2,0,0,0,1,0,1,1
5,0,0,10,347,3,0,3,0,0,0
6,0,4,23,3,7,0,62,0,5,0
7,0,4,2,4,6,0,256,0,2,2


In [39]:
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score

valid = news_df["cluster_id"] != -1
h = homogeneity_score(news_df.loc[valid, "category"], news_df.loc[valid, "cluster_id"])
c = completeness_score(news_df.loc[valid, "category"], news_df.loc[valid, "cluster_id"])
v = v_measure_score(news_df.loc[valid, "category"], news_df.loc[valid, "cluster_id"])
print(f"Homogeneity: {h:.4f}, Completeness: {c:.4f}, V-measure: {v:.4f}")

Homogeneity: 0.5006, Completeness: 0.1663, V-measure: 0.2496


In [40]:
purity_per_cluster = (
    news_df[news_df["cluster_id"] != -1]
    .groupby("cluster_id")["category"]
    .agg(lambda x: x.value_counts(normalize=True).max())
)
print(purity_per_cluster.describe())

count    633.000000
mean       0.653094
std        0.213963
min        0.196970
25%        0.465517
50%        0.671053
75%        0.843137
max        1.000000
Name: category, dtype: float64


## 4. User preference profile — recency-weighted vector over cluster_id
Built from TRAIN history only.

In [43]:
news_cluster_map = news_df.set_index("news_id")["cluster_id"].to_dict()

def build_user_profile(history_list, decay=0.9):
    """More recent clicks in history weighted higher."""
    profile = defaultdict(float)
    n = len(history_list)
    for i, nid in enumerate(history_list):
        cid = news_cluster_map.get(nid, -1)
        if cid == -1:
            continue
        weight = decay ** (n - i - 1)  # most recent click gets weight ~1
        profile[cid] += weight
    total = sum(profile.values()) or 1.0
    return {k: v / total for k, v in profile.items()}

user_histories = train_behav_df.groupby("user_id")["history_list"].last().to_dict()
user_profiles = {uid: build_user_profile(hist) for uid, hist in user_histories.items()}

## 5. Item-based & User-based Collaborative Filtering — implicit ALS
Trained on TRAIN positive clicks only (label == 1), as implicit confidence signal.

In [44]:
clicks_df = interactions_df[interactions_df["label"] == 1].copy()

user_cat = clicks_df["user_id"].astype("category")
item_cat = clicks_df["news_id"].astype("category")
clicks_df["user_idx"] = user_cat.cat.codes
clicks_df["item_idx"] = item_cat.cat.codes

user_id_lookup = dict(enumerate(user_cat.cat.categories))
item_id_lookup = dict(enumerate(item_cat.cat.categories))
user_idx_lookup = {v: k for k, v in user_id_lookup.items()}
item_idx_lookup = {v: k for k, v in item_id_lookup.items()}

conf = 1.0  # base confidence weight for a click; scale up if you add dwell-time later
user_item_matrix = sp.coo_matrix(
    (np.full(len(clicks_df), conf), (clicks_df["user_idx"], clicks_df["item_idx"]))
).tocsr()

als_model = implicit.als.AlternatingLeastSquares(factors=64, regularization=0.05, iterations=20)
als_model.fit(user_item_matrix)

def item_based_candidates(news_id, k=20):
    """Twin articles — nearest neighbors in item-factor space."""
    if news_id not in item_idx_lookup:
        return []
    idx = item_idx_lookup[news_id]
    ids, scores = als_model.similar_items(idx, N=k + 1)
    return [(item_id_lookup[i], s) for i, s in zip(ids, scores) if item_id_lookup[i] != news_id]

def user_based_candidates(user_id, k=20):
    """Articles liked by similar users, filtered to unseen items."""
    if user_id not in user_idx_lookup:
        return []
    idx = user_idx_lookup[user_id]
    ids, scores = als_model.recommend(idx, user_item_matrix[idx], N=k, filter_already_liked_items=True)
    return [(item_id_lookup[i], s) for i, s in zip(ids, scores)]

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

## 6. Bridge Apriori — FP-Growth on per-session cluster co-occurrence
Transactions = clusters clicked together within the same TRAIN impression session.

In [45]:
session_clusters = (
    interactions_df[interactions_df["label"] == 1]
    .assign(cluster_id=lambda d: d["news_id"].map(news_cluster_map))
    .groupby("impression_id")["cluster_id"]
    .apply(lambda s: list(set(c for c in s if c != -1)))
)
transactions = [t for t in session_clusters if len(t) >= 2]

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
trans_df = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = fpgrowth(trans_df, min_support=0.001, use_colnames=True)
bridge_rules = association_rules(frequent_itemsets, metric="lift", min_threshold=2.0)
bridge_rules = bridge_rules[bridge_rules["confidence"] >= 0.1].sort_values("lift", ascending=False)

print(f"{len(bridge_rules)} bridge rules found (low support, lift > 2)")
bridge_rules.head(10)

52 bridge rules found (low support, lift > 2)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
36,(465),(298),0.002469,0.034192,0.001050,0.425532,12.445338,1.0,0.000966,1.681221,0.921924,0.029499,0.405194,0.228127
20,(608),(550),0.010268,0.008876,0.001050,0.102302,11.525295,1.0,0.000959,1.104072,0.922709,0.058055,0.094262,0.110322
21,(550),(608),0.008876,0.010268,0.001050,0.118343,11.525295,1.0,0.000959,1.122582,0.921413,0.058055,0.109196,0.110322
140,(85),(182),0.006355,0.017858,0.001234,0.194215,10.875747,1.0,0.001121,1.218864,0.913860,0.053714,0.179564,0.131666
83,(118),(185),0.010977,0.012973,0.001497,0.136364,10.511318,1.0,0.001354,1.142873,0.914908,0.066667,0.125012,0.125874
82,(185),(118),0.012973,0.010977,0.001497,0.115385,10.511318,1.0,0.001354,1.118026,0.916758,0.066667,0.105566,0.125874
73,(405),(130),0.013026,0.010715,0.001392,0.106855,9.972856,1.0,0.001252,1.107642,0.911602,0.062280,0.097182,0.118378
72,(130),(405),0.010715,0.013026,0.001392,0.129902,9.972856,1.0,0.001252,1.134326,0.909472,0.062280,0.118419,0.118378
139,(382),(298),0.009402,0.034192,0.003178,0.337989,9.885005,1.0,0.002856,1.458900,0.907367,0.078622,0.314552,0.215461
38,(208),(287),0.006828,0.022427,0.001339,0.196154,8.746302,1.0,0.001186,1.216119,0.891755,0.047977,0.177712,0.127936


## 7. Content-based similarity — cold start for zero-history users/new articles

In [46]:
def content_based_candidates(user_id, k=20, exclude=set()):
    profile = user_profiles.get(user_id)
    if not profile:
        return []
    cluster_embeds = {}
    for cid in profile:
        mask = news_df["cluster_id"] == cid
        if mask.any():
            cluster_embeds[cid] = title_embeddings[mask.values].mean(axis=0)
    if not cluster_embeds:
        return []
    user_vec = np.average(
        [cluster_embeds[c] for c in profile], axis=0, weights=[profile[c] for c in profile]
    ).reshape(1, -1)
    sims = cosine_similarity(user_vec, title_embeddings).flatten()
    ranked = np.argsort(-sims)
    out = []
    for i in ranked:
        nid = news_df.iloc[i]["news_id"]
        if nid in exclude:
            continue
        out.append((nid, float(sims[i])))
        if len(out) >= k:
            break
    return out

## 8. Score combination + freshness decay

In [47]:
LAMBDA = 0.05  # tune this — larger = faster decay

def freshness_weight(article_time, now):
    age_hours = (now - article_time).total_seconds() / 3600.0
    return np.exp(-LAMBDA * max(age_hours, 0))

def blended_score(user_id, now, k_item=5, k_user=3, k_bridge=2):
    seen = set(user_histories.get(user_id, []))
    item_c = item_based_candidates(next(iter(seen), None)) if seen else []
    user_c = user_based_candidates(user_id)
    profile = user_profiles.get(user_id, {})
    bridge_clusters = set()
    for _, row in bridge_rules.iterrows():
        if set(row["antecedents"]) & set(profile.keys()):
            bridge_clusters |= set(row["consequents"])
    bridge_c = news_df[news_df["cluster_id"].isin(bridge_clusters)]["news_id"].head(k_bridge * 3).tolist()

    slots = []
    slots += [nid for nid, _ in item_c[:k_item]]
    slots += [nid for nid, _ in user_c[:k_user]]
    slots += bridge_c[:k_bridge]
    if not slots:  # cold start fallback
        slots = [nid for nid, _ in content_based_candidates(user_id, k=10, exclude=seen)]

    scored = []
    for nid in slots:
        row = news_df[news_df["news_id"] == nid]
        if row.empty:
            continue
        t = interactions_df.loc[interactions_df["news_id"] == nid, "time"]
        art_time = t.iloc[0] if len(t) else now
        scored.append((nid, freshness_weight(art_time, now)))
    return sorted(scored, key=lambda x: -x[1])

## 9. Diversity re-ranking — MMR

In [48]:
def mmr_rerank(candidates, embeddings_lookup, lambda_param=0.7, k=10):
    """candidates: list of (news_id, relevance_score)"""
    selected, pool = [], candidates.copy()
    while pool and len(selected) < k:
        if not selected:
            best = max(pool, key=lambda x: x[1])
        else:
            def mmr_score(c):
                nid, rel = c
                sim_to_selected = max(
                    cosine_similarity(
                        embeddings_lookup[nid].reshape(1, -1),
                        embeddings_lookup[s[0]].reshape(1, -1)
                    )[0][0] for s in selected
                )
                return lambda_param * rel - (1 - lambda_param) * sim_to_selected
            best = max(pool, key=mmr_score)
        selected.append(best)
        pool.remove(best)
    return selected

news_idx_lookup = {nid: i for i, nid in enumerate(news_df["news_id"])}
embed_lookup = {nid: title_embeddings[i] for nid, i in news_idx_lookup.items()}

## 10. Evaluation — on the held-out VALIDATION split
Standard MIND ranking metrics (AUC, MRR, nDCG@5, nDCG@10) computed per impression,
plus diversity metrics (ILD, Coverage, Novelty, Serendipity@k).

In [49]:
def dcg_at_k(labels, k):
    labels = np.array(labels)[:k]
    return np.sum((2 ** labels - 1) / np.log2(np.arange(2, len(labels) + 2)))

def ndcg_at_k(labels, k):
    ideal = sorted(labels, reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg_at_k(labels, k) / idcg if idcg > 0 else 0.0

def mrr(labels):
    for i, l in enumerate(labels):
        if l == 1:
            return 1.0 / (i + 1)
    return 0.0

def evaluate_impressions(eval_df, sample_n=2000):
    aucs, mrrs, ndcg5, ndcg10 = [], [], [], []
    sample = eval_df.sample(min(sample_n, len(eval_df)), random_state=42)
    for _, r in sample.iterrows():
        pairs = r["impression_list"]
        if len(pairs) < 2 or len(set(l for _, l in pairs)) < 2:
            continue
        prof = user_profiles.get(r["user_id"])
        if not prof:
            continue
        scored = []
        for nid, label in pairs:
            i = news_idx_lookup.get(nid)
            if i is None:
                continue
            cid = news_df.iloc[i]["cluster_id"]
            rel = prof.get(cid, 0.0)
            scored.append((rel, label))
        if len(scored) < 2:
            continue
        scored.sort(key=lambda x: -x[0])
        labels_ranked = [l for _, l in scored]
        scores_only = [s for s, _ in scored]
        try:
            aucs.append(roc_auc_score(labels_ranked, scores_only))
        except ValueError:
            pass
        mrrs.append(mrr(labels_ranked))
        ndcg5.append(ndcg_at_k(labels_ranked, 5))
        ndcg10.append(ndcg_at_k(labels_ranked, 10))
    return {
        "AUC": np.mean(aucs) if aucs else None,
        "MRR": np.mean(mrrs),
        "nDCG@5": np.mean(ndcg5),
        "nDCG@10": np.mean(ndcg10),
    }

print(evaluate_impressions(val_behav_df))

{'AUC': np.float64(0.550392687882216), 'MRR': np.float64(0.2923003169441898), 'nDCG@5': np.float64(0.2688474124657982), 'nDCG@10': np.float64(0.3280382500066094)}


In [54]:
def score_article_for_user(user_id, news_id):
    """Combine ALS item-affinity + content-based similarity for one article.
    Used as the ranking signal for evaluation — a stronger proxy for the
    full blended_score() than raw cluster affinity alone."""
    als_score = 0.0
    if user_id in user_idx_lookup and news_id in item_idx_lookup:
        u_idx, i_idx = user_idx_lookup[user_id], item_idx_lookup[news_id]
        als_score = float(als_model.user_factors[u_idx] @ als_model.item_factors[i_idx])
 
    content_score = 0.0
    profile = user_profiles.get(user_id)
    i = news_idx_lookup.get(news_id)
    if profile and i is not None:
        cid = news_df.iloc[i]["cluster_id"]
        content_score = profile.get(cid, 0.0)
 
    return 0.7 * als_score + 0.3 * content_score
 
def evaluate_impressions(eval_df, sample_n=2000):
    aucs, mrrs, ndcg5, ndcg10 = [], [], [], []
    sample = eval_df.sample(min(sample_n, len(eval_df)), random_state=42)
    for _, r in sample.iterrows():
        pairs = r["impression_list"]
        if len(pairs) < 2 or len(set(l for _, l in pairs)) < 2:
            continue
        prof = user_profiles.get(r["user_id"])
        if not prof:
            continue
        scored = []
        for nid, label in pairs:
            i = news_idx_lookup.get(nid)
            if i is None:
                continue
            rel = score_article_for_user(r["user_id"], nid)
            scored.append((rel, label))
        if len(scored) < 2:
            continue
        scored.sort(key=lambda x: -x[0])
        labels_ranked = [l for _, l in scored]
        scores_only = [s for s, _ in scored]
        try:
            aucs.append(roc_auc_score(labels_ranked, scores_only))
        except ValueError:
            pass
        mrrs.append(mrr(labels_ranked))
        ndcg5.append(ndcg_at_k(labels_ranked, 5))
        ndcg10.append(ndcg_at_k(labels_ranked, 10))
    return {
        "AUC": np.mean(aucs) if aucs else None,
        "MRR": np.mean(mrrs),
        "nDCG@5": np.mean(ndcg5),
        "nDCG@10": np.mean(ndcg10),
    }
 
print(evaluate_impressions(val_behav_df))
 

{'AUC': np.float64(0.5700214541403889), 'MRR': np.float64(0.29901716895957436), 'nDCG@5': np.float64(0.27790322581591037), 'nDCG@10': np.float64(0.3328390876972549)}


In [56]:
#evaluation here
def score_popularity(user_id, news_id):
    return click_counts.get(news_id, 0)

def evaluate_baseline(eval_df, score_fn, sample_n=2000):
    aucs, mrrs, ndcg5, ndcg10 = [], [], [], []
    sample = eval_df.sample(min(sample_n, len(eval_df)), random_state=42)
    for _, r in sample.iterrows():
        pairs = r["impression_list"]
        if len(pairs) < 2 or len(set(l for _, l in pairs)) < 2:
            continue
        scored = [(score_fn(r["user_id"], nid), label) for nid, label in pairs]
        scored.sort(key=lambda x: -x[0])
        labels_ranked = [l for _, l in scored]
        scores_only = [s for s, _ in scored]
        try:
            aucs.append(roc_auc_score(labels_ranked, scores_only))
        except ValueError:
            pass
        mrrs.append(mrr(labels_ranked))
        ndcg5.append(ndcg_at_k(labels_ranked, 5))
        ndcg10.append(ndcg_at_k(labels_ranked, 10))
    return {"AUC": np.mean(aucs) if aucs else None, "MRR": np.mean(mrrs),
            "nDCG@5": np.mean(ndcg5), "nDCG@10": np.mean(ndcg10)}

print(evaluate_baseline(val_behav_df, score_popularity))

{'AUC': np.float64(0.5225668626827616), 'MRR': np.float64(0.2509322387858467), 'nDCG@5': np.float64(0.22328601871413162), 'nDCG@10': np.float64(0.2897608896838426)}


In [57]:
val_interactions = flatten_impressions(val_behav_df)
users_seen = val_interactions["user_id"].isin(user_idx_lookup.keys())
items_seen = val_interactions["news_id"].isin(item_idx_lookup.keys())
print(f"Val users seen in training: {users_seen.mean():.1%}")
print(f"Val items seen in training: {items_seen.mean():.1%}")

Val users seen in training: 86.2%
Val items seen in training: 70.6%


In [50]:
def intra_list_diversity(news_ids):
    if len(news_ids) < 2:
        return 0.0
    vecs = np.array([embed_lookup[n] for n in news_ids if n in embed_lookup])
    if len(vecs) < 2:
        return 0.0
    sims = cosine_similarity(vecs)
    n = len(vecs)
    return 1 - (sims.sum() - n) / (n * (n - 1))  # exclude diagonal

def coverage(recommended_sets, catalog_size):
    all_recommended = set().union(*recommended_sets) if recommended_sets else set()
    return len(all_recommended) / catalog_size

def novelty(recommended_sets, click_counts):
    scores = []
    total_clicks = sum(click_counts.values()) or 1
    for rec_set in recommended_sets:
        for nid in rec_set:
            p = click_counts.get(nid, 0) / total_clicks
            if p > 0:
                scores.append(-np.log2(p))
    return np.mean(scores) if scores else 0.0

click_counts = interactions_df[interactions_df["label"] == 1]["news_id"].value_counts().to_dict()

## 11. Save artifacts for download

In [51]:
os.makedirs("/kaggle/working/artifacts", exist_ok=True)

topic_model.save("/kaggle/working/artifacts/bertopic_model")
np.save("/kaggle/working/artifacts/title_embeddings.npy", title_embeddings)
news_df.to_parquet("/kaggle/working/artifacts/news_with_clusters.parquet")
bridge_rules.to_json("/kaggle/working/artifacts/bridge_rules.json", orient="records")

sp.save_npz("/kaggle/working/artifacts/user_item_matrix.npz", user_item_matrix)
np.save("/kaggle/working/artifacts/als_user_factors.npy", als_model.user_factors)
np.save("/kaggle/working/artifacts/als_item_factors.npy", als_model.item_factors)

import json
with open("/kaggle/working/artifacts/user_id_lookup.json", "w") as f:
    json.dump({str(k): v for k, v in user_id_lookup.items()}, f)
with open("/kaggle/working/artifacts/item_id_lookup.json", "w") as f:
    json.dump({str(k): v for k, v in item_id_lookup.items()}, f)

print("Artifacts saved to /kaggle/working/artifacts — commit the notebook to download them.")

2026-08-03 11:11:18,523 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Artifacts saved to /kaggle/working/artifacts — commit the notebook to download them.
